In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt   

In [ ]:
def smoothabs(x, xi=10000):
    """Smooth absolute value function"""
    return x * np.tanh(x * xi)

# model parameters, variable
#loop T by 0.01, from 0 to 1
T0 = np.linspace(0., 1., 101)                        # value of temperature at which we are computing the covariance matrix
sigma_0=0.81
eps_f=3*0.0001
F_1=0.1
x_avg=1.0147
delta_avg=1.7463

# numerical parameters
dt=7.5*0.000001               # fast timestep
dt_slow=dt/eps_f              # slow timestep
N_trans=int(0.1/dt)           # number of timesteps in transient to let fast variables equilibrate
N_stat_slow=int(1/dt_slow)  # number of timesteps for stationary run where we analyse Brownian motion
N_fast_slow=int(dt_slow/dt)   # number of fast timesteps within one slow timestep
M=100                         # ensemble size

# model parameters, fixed
a=0.25
b=4.
F_0=8.
G_0=1.
eps_a=0.34
mu=7.5
theta_0=1.
theta_1=0.0195
sigma_1=0.00934

# initialization
x=np.zeros((N_fast_slow,M))
y=np.zeros((N_fast_slow,M))
z=np.zeros((N_fast_slow,M))
T=np.zeros((N_stat_slow,M))
S=np.zeros((N_stat_slow,M))

results = []   # list of DataFrames, one per t0
correlations = []

In [ ]:
'''
For a given starting temperature (t0), runs a transient until the fast variables reach their attractor, then uses separation of 
time-scales to allow fast, chaotic atmosphere to run while the slow ocean is essentially stationary. The integral of the fast
atmosphere is used to act like noise on the slower system. We observe the variance of T and S due to noise, and confirm that
it can be approximated with white noise given that the approximately random forcing from the atmosphere leads to, on average
a continuous spread of trajectories over time, shown in the plots as an approximately straight line from 0 to 1. 
'''
def stochastic_system(t0):
# transient - here we only run the fast variables for a relatively short time from random initial conditions
# so that they equilibrate on their attractor for the given value of T0
    for m in range(M):
    
        # the fast variables are initialised randomly near the average value
        x[0,m]=x_avg*(1+0.1*np.random.randn())
        y[0,m]=np.sqrt(delta_avg/2)*(1+0.1*np.random.randn())
        z[0,m]=np.sqrt(delta_avg/2)*(1+0.1*np.random.randn())
    
        # counter for testing, remove it later
        #print(m)
        
        for n in range(N_trans-1):
    
            x_now=x[0,m]
            y_now=y[0,m]
            z_now=z[0,m]
            
            k1x = 1/eps_f*(-(y_now**2+z_now**2) - a*(x_now - F_0 - F_1*t0))
            k1y = 1/eps_f*(x_now*y_now - b*x_now*z_now - (y_now - G_0))
            k1z = 1/eps_f*(b*x_now*y_now + x_now*z_now - z_now)  
            
            x1=x_now+k1x*dt/2
            y1=y_now+k1y*dt/2
            z1=z_now+k1z*dt/2
            
            k2x = 1/eps_f*(-(y1**2+z1**2) - a*(x1 - F_0 - F_1*t0))
            k2y = 1/eps_f*(x1*y1 - b*x1*z1 - (y1 - G_0))
            k2z = 1/eps_f*(b*x1*y1 + x1*z1 - z1)
         
            x2=x_now+k2x*dt/2
            y2=y_now+k2y*dt/2
            z2=z_now+k2z*dt/2     
    
            k3x = 1/eps_f*(-(y2**2+z2**2) - a*(x2 - F_0 - F_1*t0))
            k3y = 1/eps_f*(x2*y2 - b*x2*z2 - (y2 - G_0))
            k3z = 1/eps_f*(b*x2*y2 + x2*z2 - z2)
    
            x3=x_now+k3x*dt
            y3=y_now+k3y*dt
            z3=z_now+k3z*dt
    
            k4x = 1/eps_f*(-(y3**2+z3**2) - a*(x3 - F_0 - F_1*t0))
            k4y = 1/eps_f*(x3*y3 - b*x3*z3 - (y3 - G_0))
            k4z = 1/eps_f*(b*x3*y3 + x3*z3 - z3)
            
            x[0,m]=x[0,m]+(k1x+2*k2x+2*k3x+k4x)*dt/6
            y[0,m]=y[0,m]+(k1y+2*k2y+2*k3y+k4y)*dt/6
            z[0,m]=z[0,m]+(k1z+2*k2z+2*k3z+k4z)*dt/6

    # stationary - here we run the slow variables as a Brownian motion: at each dt_slow, we update them 
    # with the integral of the coupling term calculated from a run of the fast variables within dt_slow 
    for m in range(M):
    
        # the slow variables are initialised at zero because we only want 
        # to see the growth of the variance on the Brownian motion
        T[0,m]=0
        S[0,m]=0
        
        # counter for testing, remove it later
        #print(m)
    
        for n in range(N_stat_slow-1):
    
            # evolution of the fast variables for a time dt_slow
            for j in range(N_fast_slow-1):
    
                x_now=x[j,m]
                y_now=y[j,m]
                z_now=z[j,m]
        
                k1x = 1/eps_f*(-(y_now**2+z_now**2) - a*(x_now - F_0 - F_1*t0))
                k1y = 1/eps_f*(x_now*y_now - b*x_now*z_now - (y_now - G_0))
                k1z = 1/eps_f*(b*x_now*y_now + x_now*z_now - z_now)  
            
                x1=x_now+k1x*dt/2
                y1=y_now+k1y*dt/2
                z1=z_now+k1z*dt/2
           
                k2x = 1/eps_f*(-(y1**2+z1**2) - a*(x1 - F_0 - F_1*t0))
                k2y = 1/eps_f*(x1*y1 - b*x1*z1 - (y1 - G_0))
                k2z = 1/eps_f*(b*x1*y1 + x1*z1 - z1)
            
                x2=x_now+k2x*dt/2
                y2=y_now+k2y*dt/2
                z2=z_now+k2z*dt/2     
    
                k3x = 1/eps_f*(-(y2**2+z2**2) - a*(x2 - F_0 - F_1*t0))
                k3y = 1/eps_f*(x2*y2 - b*x2*z2 - (y2 - G_0))
                k3z = 1/eps_f*(b*x2*y2 + x2*z2 - z2)
           
                x3=x_now+k3x*dt
                y3=y_now+k3y*dt
                z3=z_now+k3z*dt
        
                k4x = 1/eps_f*(-(y3**2+z3**2) - a*(x3 - F_0 - F_1*t0))
                k4y = 1/eps_f*(x3*y3 - b*x3*z3 - (y3 - G_0))
                k4z = 1/eps_f*(b*x3*y3 + x3*z3 - z3)
            
                x[j+1,m]=x[j,m]+(k1x+2*k2x+2*k3x+k4x)*dt/6
                y[j+1,m]=y[j,m]+(k1y+2*k2y+2*k3y+k4y)*dt/6
                z[j+1,m]=z[j,m]+(k1z+2*k2z+2*k3z+k4z)*dt/6
    
            # evolution of the slow variables: the integrated effect of the fast variables should behave
            # like a Gaussian noise
            T[n+1,m]=T[n,m]+np.sum(1/eps_a*theta_1*(x[:,m]-x_avg)/np.sqrt(eps_f))*dt
            S[n+1,m]=S[n,m]+np.sum(sigma_1*(y[:,m]**2+z[:,m]**2-delta_avg)/np.sqrt(eps_f))*dt
            
            x[0,m]=x[-1,m]
            y[0,m]=y[-1,m]
            z[0,m]=z[-1,m]
  
    return S,T

In [ ]:
for t0 in T0:
    S,T = stochastic_system(t0)
    time=np.linspace(0,(N_stat_slow-1)*dt_slow,N_stat_slow)
    # create a long DataFrame: columns = ['t0','time','member','T','S']
    nsteps, nmem = T.shape
    df = pd.DataFrame({
        't0': np.repeat(t0, nsteps * nmem),
        'time': np.tile(np.repeat(time, nmem), 1),          # length nsteps*nmem
        'member': np.tile(np.arange(nmem), nsteps),
        'T': T.flatten(order='C'),    # row-major: time-major then member
        'S': S.flatten(order='C'),
    })
    results.append(df)


In [ ]:
#concat dfs
#big_df = pd.concat(results, ignore_index=True)
#big_df.tail()
big_df.to_csv('big_df.csv',index=False)

In [ ]:
print(len(big_df))

In [ ]:
initial_t = big_df['t0'].unique().tolist()
#loop
for temp in initial_t: 
    #filter based on value of t0
    df=big_df.loc[big_df['t0']==temp]
    
    plt.figure()
    sns.lineplot(data=df, x='time', y='T', palette=pal, hue='member',
                 estimator=None, lw=0.6, legend=False, alpha=0.6, zorder=1)
    t_mean = df.groupby("time", as_index=False)['T'].mean()
        
    sns.lineplot(data=t_mean, x='time', y='T', color='#2b2b2b', linewidth=2,
                 label='Temperature mean', zorder=2,alpha=1)
    plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.6)
    plt.ylabel('Temperature')
    plt.title(f"Timeseries of Temperature, T0={temp}")
    plt.show()
    t_var = df.groupby("time",as_index=False)['T'].var()
    #variance
    plt.figure()
    sns.lineplot(data=t_var, x='time', y='T', color='#2b2b2b', linewidth=2,
                 label='Temperature mean', zorder=2,alpha=1)
    plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.6)
    plt.ylabel('Temperature')
    plt.title(f"Variance of Temperature, T0={temp}")
    plt.show()
    
    plt.figure()
    
    sns.lineplot(data=df, x='time', y='S',  palette=pal, hue='member',
                 estimator=None, lw=0.6, legend=False, alpha=0.4, zorder=1)
  
    s_mean = df.groupby("time", as_index=False)['S'].mean()
    s_var = df.groupby("time",as_index=False)['S'].var()
    sns.lineplot(data=s_mean, x='time', y='S', color='#2b2b2b', linewidth=2,
                  zorder=2,alpha=1)
    plt.ylabel('Salinity')
    plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.4)
    plt.title(f"Timeseries of Salinity, T0={temp}")
    plt.show()

    #variance
    plt.figure()
    sns.lineplot(data=s_var, x='time', y='S', color='#2b2b2b', linewidth=2,
                 zorder=2,alpha=1)
    plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.6)
    plt.ylabel('Salinity')
    plt.title(f"Variance of Salinity, T0={temp}")
    plt.show()

    #covariance matrix
    cov_TS = df[['T','S']].cov()
    print(cov_TS)

    #covariance plot
    cov_t = df.groupby('time').apply(lambda g: g['T'].cov(g['S']))
    plt.figure()
    sns.lineplot(x=cov_t.index, y=cov_t.values, color='#2b2b2b')
    plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.6)
    plt.ylabel('Covariance(T,S)'); plt.title(f'Covariance vs time, t0={temp}')
    plt.show()


In [ ]:
g = big_df.groupby(['t0','time'])
varT_grid = g['T'].var().unstack('time')          # rows=t0, cols=time
varS_grid = g['S'].var().unstack('time')
covTS_grid = g.apply(lambda d: d['T'].cov(d['S'])).unstack('time')
corrTS_grid = covTS_grid / np.sqrt(varT_grid * varS_grid)
sns.heatmap(corrTS_grid, cmap='coolwarm', center=0)
plt.xlabel('time'); plt.ylabel('t0'); plt.title('Corr(T,S) vs time and t0'); plt.show()

#prepare data to plot slope
time_vals = varT_grid.columns.values
slope = lambda y: np.polyfit(time_vals, y, 1)[0]
slope_T = varT_grid.apply(slope, axis=1)
slope_S = varS_grid.apply(slope, axis=1)
slope_TS = covTS_grid.apply(slope, axis=1)
sigma_T = np.sqrt(np.clip(slope_T, 0, None))
sigma_S = np.sqrt(np.clip(slope_S, 0, None))
corr_TS = slope_TS / np.sqrt(slope_T * slope_S)
df_slopes = pd.DataFrame({'t0': slope_T.index.values,'sigma_T': sigma_T, 'sigma_S': sigma_S, 'corr_TS': corr_TS})

In [ ]:
df_slopes.head()

In [ ]:
'''
This is a bad plot but it does show that T and S are strongly anti-correlated regardless of the initial values of temperature.
'''

# one row (metric), columns = t0 values
corr_grid = (df_slopes.set_index('t0')['corr_TS']       # index = t0, values = corr
                            .to_frame()                 # make it a 1-column DF
                            .T)                         # transpose → 1 × nT matrix

plt.figure(figsize=(10, 2))
sns.heatmap(corr_grid,
            cmap="coolwarm", vmin=-1, vmax=1, center=0,
            xticklabels=[f'{t0:.2f}' for t0 in corr_grid.columns],
            yticklabels=['corr_TS'], annot=True)

plt.xlabel('t0')
plt.ylabel('')
plt.title('Correlation between T and S for each t0')
plt.tight_layout()
plt.show()


In [ ]:
'''
First two plots are how noise strength depends on temperature, T0 (similar for S and T since they are highly correlated)
3rd plot is how the correlation of noises depends on temperature
'''

plt.figure()
sns.lineplot(data=df_slopes,x='t0',y='sigma_T')
plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.6)
plt.ylabel("")
plt.title("Noise Amplitude on Temperature for T0")
plt.show()
plt.figure()
sns.lineplot(data=df_slopes,x='t0',y='sigma_S')
plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.6)
plt.ylabel("")
plt.title("Noise Amplitude on Salinity for T0")
plt.show()
plt.figure()
sns.lineplot(data=df_slopes,x='t0',y='corr_TS')
plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.6)
plt.ylabel("")
plt.title("Estimated Correlation Coefficient Between Noise in S and T")
plt.show()

In [ ]:
'''
Results in a dataframe that I will need to compute the SDE
t0: all the initial conditions of temperature
sigma_T/sigma_S: effective variances per unit time, effectively noise amplitude
corr_TS: covariance per unit time, so correlation
xavg and delta avg are the drift terms, how much centering constants should be corrected based on linear trend of ensemble
mean over time
So the final equation will add the drift term, and still add random Guassian noise, but multiply it by a coefficient equal
to the noise amplitude
'''
#prepare data for deviation of avg of x
meanT_grid = big_df.groupby(['t0','time'])['T'].mean().unstack('time')
meanS_grid = big_df.groupby(['t0','time'])['S'].mean().unstack('time')

time_vals = meanT_grid.columns.values
get_slope = lambda y: np.polyfit(time_vals, y, 1)[0]
slope_Tmean = meanT_grid.apply(get_slope, axis=1)
slope_Smean = meanS_grid.apply(get_slope, axis=1)
x_avg_corr = slope_Tmean*(np.sqrt(eps_f)*eps_a/theta_1) / (N_fast_slow*dt)
delta_avg_corr = slope_Smean*(np.sqrt(eps_f)/sigma_1) / (N_fast_slow*dt)

df_slopes = pd.DataFrame({
    't0': slope_Tmean.index.values,
    'sigma_T': sigma_T.values,           
    'sigma_S': sigma_S.values,
    'corr_TS': corr_TS.values,
    'x_avg_corr': x_avg_corr.values,
    'delta_avg_corr': delta_avg_corr.values
}).sort_values('t0')

In [ ]:
df_slopes.head()
df_slopes.to_csv('../data/SDE_terms.csv',index=False)

In [ ]:

'''
scratch paper: How F did the variance and ensemble means:
# calculate the terms of the covariance of the ensemble 
    C11=np.zeros((N_stat_slow,))
    C22=np.zeros((N_stat_slow,))
    C12=np.zeros((N_stat_slow,))
    for t in range(N_stat_slow):
        C11[t]=np.mean((T[t,:]-np.mean(T[t,:]))*(T[t,:]-np.mean(T[t,:])))
        C22[t]=np.mean((S[t,:]-np.mean(S[t,:]))*(S[t,:]-np.mean(S[t,:])))
        C12[t]=np.mean((T[t,:]-np.mean(T[t,:]))*(S[t,:]-np.mean(S[t,:])))

Plots of F:
# plot ensembles of slow variables
time=np.linspace(0,(N_stat_slow-1)*dt_slow,N_stat_slow)
plt.figure()
for i in range(M):
    plt.plot(time,T[:,i],'b-')
plt.plot(time,np.mean(T,axis=1),'r-')
title=f"Ensemble and Mean of Temperature, T0={t0}"
plt.title(title)
plt.figure()
for i in range(M):
    plt.plot(time,S[:,i],'b-')
plt.plot(time,np.mean(S,axis=1),'r-')
title=f"Ensemble and Mean of Salinity, T0={t0}"
plt.title(title)

# plot the growth of the variances and of the covariance
time=np.linspace(0,(N_stat_slow-1)*dt_slow,N_stat_slow)
plt.figure()
plt.plot(time,C11)
title=f"Variance of Temperature, T0={t0}"
plt.title(title)
plt.figure()
plt.plot(time,C22)
title=f"Variance of Salinity, T0={t0}"
plt.title(title)
plt.figure()
plt.plot(time,C12)
title=f"Co-variance of Temperature and Salinity, T0={t0}"
plt.title(title)

# calculate variances and covariance of the noise with a linear fit
time=np.linspace(0,(N_stat_slow-1)*dt_slow,N_stat_slow)
#slope intercept of variance
varT,p0=np.polyfit(time,C11,1)
varS,p0=np.polyfit(time,C22,1)
covTS,p0=np.polyfit(time,C12,1)

# print noise amplitudes and correlation
print(np.sqrt(varT))
print(np.sqrt(varS))
print(covTS/(np.sqrt(varT)*np.sqrt(varS)))

# calculate deviation of average of x and (y**2+z**2) from x_avg and delta_avg respectively 
x_avg_corr,p0=np.polyfit(time,np.mean(T,axis=1),1)*(np.sqrt(eps_f)*eps_a/theta_1)/(N_fast_slow*dt)
delta_avg_corr,p0=np.polyfit(time,np.mean(S,axis=1),1)*(np.sqrt(eps_f)/sigma_1)/(N_fast_slow*dt)
#compare fixed centering terms with actual expectations
print(x_avg,x_avg+x_avg_corr)
print(delta_avg,delta_avg+delta_avg_corr)
'''
